# Shared chance-child interface

Generate exactly one chance street at a time. Children retain their reveal, next state and exact conditional probability; no descendant showdown work happens here.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from cards import hand_names, make_hand
from chance import ChanceState, canonical_chance_children, canonical_state, chance_children

## Public versus hand-conditioned rivers

A public turn leaves 48 rivers. Conditioning on two compatible private hands blocks four more cards and leaves 44.

In [2]:
turn = make_hand(['2c', '5c', '8c', 'Jc'])
first = make_hand(['Ah', 'Kh'])
second = make_hand(['Js', 'Jd'])
public_state = ChanceState(turn)
conditioned_state = ChanceState(turn, first, second)
public = chance_children(public_state)
conditioned = chance_children(conditioned_state)

print(f'Public:      {len(public)} children at probability {public[0].probability}')
print(f'Conditioned: {len(conditioned)} children at probability {conditioned[0].probability}')

Public:      48 children at probability 1/48
Conditioned: 44 children at probability 1/44


## Conditional canonical children

The four-club turn fixes clubs and permits permutations of the other three suits. Its 48 raw rivers therefore form 22 stabilizer orbits: nine remaining clubs with multiplicity one and thirteen off-suit rank classes with multiplicity three.

In [3]:
groups = canonical_chance_children(public_state)
table = pd.DataFrame([
    {
        'representative': ' '.join(hand_names(group.representative.reveal)),
        'multiplicity': group.multiplicity,
        'probability': str(group.probability),
    }
    for group in groups
])
print(f'{len(public)} raw children -> {len(groups)} canonical groups')
table

48 raw children -> 22 canonical groups


,representative,multiplicity,probability
0,3c,1,1/48
1,4c,1,1/48
2,6c,1,1/48
3,7c,1,1/48
4,9c,1,1/48
5,Tc,1,1/48
6,Qc,1,1/48
7,Kc,1,1/48
8,Ac,1,1/48
9,2d,3,1/16


## Canonical keys and transport

The global key includes the board, ordered fixed hands, reveal history and action history. The returned permutation transports every card-bearing component into those canonical coordinates.

In [4]:
key, transport = canonical_state(conditioned_state)
print('Canonical key:', key)
print('Suit transport:', transport)

Canonical key: (585, 50331648, 281509336449024, (), ())
Suit transport: (0, 2, 1, 3)
